# 02 — SQLAlchemy Core

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre la distinction Core vs ORM
- définir un schéma avec `MetaData`, `Table`, `Column`
- construire des requêtes avec `select`, `insert`, `update`, `delete`
- utiliser les types SQLAlchemy (`Integer`, `String`, `DateTime`)
- exécuter des requêtes avec un `Engine` et une `Connection`

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- SQL avec `sqlite3` (notebook 01)
- context managers

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- SQLAlchemy ORM (notebook 03)
- Alembic (notebook 04)

## Plan

1. Core vs ORM : deux couches
2. Créer un `Engine`
3. Définir un schéma (Table, Column)
4. `insert()`
5. `select()` et filtrage
6. `update()`, `delete()`
7. Jointures
8. Synthèse
9. Exercices

---

## 1. Core vs ORM : deux couches

SQLAlchemy a **deux** API :

- **Core** (cette section) : expression language SQL, pas de classes Python mappées, contrôle total sur le SQL généré.
- **ORM** (section suivante) : classes Python mappées automatiquement aux tables, unit of work, relationships.

L'approche recommandée en SQLAlchemy 2.0 : **apprendre Core d'abord**, puis ORM au-dessus.

---

## 2. Créer un `Engine`

In [ ]:
from sqlalchemy import create_engine

engine = create_engine('sqlite:///:memory:', echo=True)  # echo pour voir le SQL généré


`echo=True` affiche les requêtes SQL dans la sortie standard — utile pour apprendre. À désactiver en production.

---

## 3. Définir un schéma

In [ ]:
from sqlalchemy import MetaData, Table, Column, Integer, String, ForeignKey

metadata = MetaData()

salle = Table(
    'salle', metadata,
    Column('id', Integer, primary_key=True, autoincrement=True),
    Column('nom', String(50), nullable=False, unique=True),
    Column('capacite', Integer, nullable=False),
)

reservation = Table(
    'reservation', metadata,
    Column('id', Integer, primary_key=True, autoincrement=True),
    Column('salle_id', Integer, ForeignKey('salle.id'), nullable=False),
    Column('creneau', String(50), nullable=False),
    Column('organisateur', String(100), nullable=False),
)

metadata.create_all(engine)


---

## 4. `insert()`

In [ ]:
from sqlalchemy import insert

with engine.begin() as conn:
    conn.execute(insert(salle), [
        {'nom': 'Mars', 'capacite': 12},
        {'nom': 'Venus', 'capacite': 6},
        {'nom': 'Io', 'capacite': 15},
    ])


---

## 5. `select()` et filtrage

In [ ]:
from sqlalchemy import select

with engine.connect() as conn:
    result = conn.execute(select(salle))
    for row in result:
        print(row._mapping)


In [ ]:
with engine.connect() as conn:
    result = conn.execute(
        select(salle).where(salle.c.capacite >= 10).order_by(salle.c.nom)
    )
    for row in result:
        print(row.nom, row.capacite)


`salle.c.nom` est un raccourci pour `salle.columns.nom`. Le `.c` est la convention Core.

---

## 6. `update()`, `delete()`

In [ ]:
from sqlalchemy import update, delete

with engine.begin() as conn:
    conn.execute(update(salle).where(salle.c.nom == 'Mars').values(capacite=14))
    conn.execute(delete(salle).where(salle.c.nom == 'Io'))


In [ ]:
with engine.connect() as conn:
    for row in conn.execute(select(salle)):
        print(row.nom, row.capacite)


---

## 7. Jointures

In [ ]:
with engine.begin() as conn:
    conn.execute(insert(reservation), [
        {'salle_id': 1, 'creneau': 'lundi 9h', 'organisateur': 'Alice'},
        {'salle_id': 2, 'creneau': 'mardi 14h', 'organisateur': 'Bob'},
    ])


In [ ]:
stmt = (
    select(salle.c.nom, reservation.c.creneau, reservation.c.organisateur)
    .join(reservation, salle.c.id == reservation.c.salle_id)
)

with engine.connect() as conn:
    for row in conn.execute(stmt):
        print(dict(row._mapping))


---

## Synthèse

| Core | Rôle |
|---|---|
| `create_engine(url)` | Connexion à la base |
| `Table(...)`, `Column(...)` | Définition du schéma |
| `insert()`, `select()`, `update()`, `delete()` | Requêtes expressives |
| `engine.begin()` | Transaction auto-commit/rollback |
| `engine.connect()` | Connexion sans auto-commit |


### Règles à retenir

1. **Core pour le contrôle**, ORM pour la productivité.
2. **`engine.begin()` pour les écritures** : auto-commit/rollback.
3. **Le SQL généré est visible avec `echo=True`** — utilisez-le pour apprendre.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — CRUD Core *(facile)*

Définir une table `produit` (id, nom, prix) en Core. Insérer 3 produits, sélectionner ceux avec prix > 10.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_SQLAlchemy_Core", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, Float, insert, select

engine = create_engine('sqlite:///:memory:')
meta = MetaData()
produit = Table('produit', meta,
    Column('id', Integer, primary_key=True),
    Column('nom', String(50)),
    Column('prix', Float))
meta.create_all(engine)

with engine.begin() as conn:
    conn.execute(insert(produit), [
        {'nom': 'pain', 'prix': 1.2},
        {'nom': 'vin', 'prix': 12.5},
        {'nom': 'fromage', 'prix': 8.0},
    ])

with engine.connect() as conn:
    for row in conn.execute(select(produit).where(produit.c.prix > 10)):
        print(dict(row._mapping))
```

</details>

### Exercice 2 — Jointure Core *(moyen)*

Ajouter une table `commande(id, produit_id, quantite)` et une jointure SELECT avec GROUP BY pour le total par produit.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_SQLAlchemy_Core", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, Float, ForeignKey, insert, select, func

engine = create_engine('sqlite:///:memory:')
meta = MetaData()
produit = Table('produit', meta,
    Column('id', Integer, primary_key=True),
    Column('nom', String(50)),
    Column('prix', Float))
commande = Table('commande', meta,
    Column('id', Integer, primary_key=True),
    Column('produit_id', Integer, ForeignKey('produit.id')),
    Column('quantite', Integer))
meta.create_all(engine)

with engine.begin() as conn:
    conn.execute(insert(produit), [{'nom': 'A', 'prix': 10}, {'nom': 'B', 'prix': 20}])
    conn.execute(insert(commande), [{'produit_id': 1, 'quantite': 5}, {'produit_id': 1, 'quantite': 3}, {'produit_id': 2, 'quantite': 7}])

stmt = select(produit.c.nom, func.sum(commande.c.quantite).label('total')).join(commande).group_by(produit.c.nom)
with engine.connect() as conn:
    for row in conn.execute(stmt):
        print(dict(row._mapping))
```

</details>

### Exercice 3 — Recherche paramétrique avec Core *(difficile)*

Écrire une fonction `chercher(engine, nom: str | None, cap_min: int | None) -> list[dict]` qui construit dynamiquement un `select` avec des `where` conditionnels.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_SQLAlchemy_Core", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from sqlalchemy import select

def chercher(engine, nom: str | None = None, cap_min: int | None = None) -> list[dict]:
    stmt = select(salle)
    if nom:
        stmt = stmt.where(salle.c.nom.ilike(f'%{nom}%'))
    if cap_min is not None:
        stmt = stmt.where(salle.c.capacite >= cap_min)
    with engine.connect() as conn:
        return [dict(r._mapping) for r in conn.execute(stmt)]

print(chercher(engine, cap_min=10))
```

</details>

---

## Ressources externes

### Documentation officielle
- [SQLAlchemy 2.0 Core Tutorial](https://docs.sqlalchemy.org/en/20/core/)
- [Expression Language](https://docs.sqlalchemy.org/en/20/core/expression_api.html)